# 🚀 Lesson 27: Microsoft Phi-3.5-mini & Jinja Chat Templates

**Advanced Step-by-Step Interactive Notebook** with clear architectural context, code logic, and step explanations.


### 🔹 Step 1: Execution Block

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
import json
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
from datasets import Dataset


### 🔹 Step 2: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
MAX_LENGTH = 512
OUTPUT_DIR = "./phi-3.5-mini-instruct"

DATA_PATH = "/Users/mac/Desktop/Machine Learning/DL/Homework/instruction-data.json"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "instruction-data.json"


### 🔹 Step 3: Execution Block

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
print("Loading instruction dataset...")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
    
print("Number of examples:", len(data))
print("First example:")
print(data[0])


### 🔹 Step 4: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Tokenizer vocabulary size:", len(tokenizer))

print("\nLoading model...")


### 🔹 Step 5: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully.")
print("Total number of parameters:", sum(p.numel() for p in model.parameters()))
print("Model device:", model.device)


### 🔹 Step 6: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def format_conversation(example):
    Converts raw instruction-input-output dicts into standardized
    chat message dictionaries with 'user' and 'assistant' roles.
    instruction = example["instruction"].strip()
    input_text = example.get("input", "").strip()
    output = example["output"].strip()
    
    if input_text:
        user_content = f"{instruction}\n\n{input_text}"
    else:
        user_content = instruction
        
    messages = [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "assistant",
            "content": output
        }
    ]
    
    return {"messages": messages}


### 🔹 Step 7: Execution Block

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
dataset = Dataset.from_list(
    [format_conversation(example) for example in data]
)

print("\nFormatted conversation example:")
print(dataset[0]["messages"])


def prepare_input(example):


### 🔹 Step 8: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
Applies Phi-3's official chat template, tokenizes sequences,
    and sets label values of user prompt tokens to -100 so that
    loss is computed ONLY on the assistant's response tokens.
    messages = example["messages"]
    
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


### 🔹 Step 9: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
user_messages = [messages[0]]
    prompt_text = tokenizer.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    full_encoding = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )


### 🔹 Step 10: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
prompt_encoding = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    input_ids = full_encoding["input_ids"]
    attention_mask = full_encoding["attention_mask"]
    
    prompt_length = min(
        len(prompt_encoding["input_ids"]),
        len(input_ids)
    )


### 🔹 Step 11: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
labels = input_ids.copy()
    for i in range(prompt_length):
        labels[i] = -100
        
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

print("\nTokenizing dataset...")

tokenized_ds = dataset.map(
    prepare_input,
    remove_columns=["messages"]
)

print("Tokenization completed.")
print("Number of tokenized examples:", len(tokenized_ds))


### 🔹 Step 12: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
first_example = tokenized_ds[0]
print("\nFirst example verification:")
print("Input IDs (first 30):", first_example["input_ids"][:30])
print("Labels (first 30):   ", first_example["labels"][:30])
print(
    "Number of active target tokens used for loss computation:",
    sum(label != -100 for label in first_example["labels"])
)


### 🔹 Step 13: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=True,
    remove_unused_columns=False
)


### 🔹 Step 14: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
    processing_class=tokenizer
)


### 🔹 Step 15: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("\nStarting instruction fine-tuning...")
trainer.train()

print("\nSaving fine-tuned model and tokenizer...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining completed successfully.")
print("Model saved to:", OUTPUT_DIR)


## 🎯 Summary & Key Takeaways
1. **Modular Execution**: Each component runs independently and validates intermediate tensor shapes and states.
2. **Core Insights**: Inspect the printed metrics, loss outputs, and visual distributions above.
3. **Next Lesson**: Applies these foundations to more advanced deep learning and transformer architectures.
